# 04 — Run MEFISTO once on the full cohort

**Input:** the same filtered counts and metadata used by TEMPTED  
**Does:** applies pseudocount-free rCLR and fits one five-factor MEFISTO model using all eligible subjects  
**Output:** fitted sample factors, subject mean factor scores, genus loadings, and the saved model

Original zeros stay missing (`NaN`). There is no train/test split or held-out projection here.

In [1]:
from datetime import datetime
from pathlib import Path

import anndata as ad
import muon as mu
import numpy as np
import pandas as pd

N_FACTORS = 5
N_ITERATIONS = 1000
SEED = 2026

root = Path(".") if Path("data").exists() else Path("..")
input_folder = sorted((root / "data" / "preprocessing").iterdir())[-1]
output = root / "data" / "mefisto" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

counts = pd.read_csv(input_folder / "counts_filtered.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(
    input_folder / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str, "country": str},
)
counts = counts.loc[metadata["sample_id"]]

print("Input:", input_folder)
print("Output:", output)

Input: ../data/preprocessing/20260811_023248
Output: ../data/mefisto/20260811_023339


/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/venv/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


## rCLR

Only positive counts are logged and centered within each sample. Zeros remain missing.

In [2]:
values = counts.to_numpy(float)
rclr = np.full(values.shape, np.nan)

for i, row in enumerate(values):
    positive = row > 0
    logged = np.log(row[positive])
    rclr[i, positive] = logged - logged.mean()

rclr = pd.DataFrame(rclr, index=counts.index, columns=counts.columns)

## Fit one full-cohort MEFISTO model

In [3]:
obs = metadata.set_index("sample_id")[["subject_id", "age", "country"]].copy()
obs["model_group"] = pd.factorize(obs["subject_id"])[0].astype(str)

adata = ad.AnnData(
    rclr.loc[obs.index].to_numpy(np.float32),
    obs=obs,
    var=pd.DataFrame(index=rclr.columns.astype(str)),
)

mu.tl.mofa(
    adata,
    groups_label="model_group",
    likelihoods="gaussian",
    center_groups=False,
    n_factors=N_FACTORS,
    n_iterations=N_ITERATIONS,
    convergence_mode="medium",
    smooth_covariate="age",
    smooth_kwargs={"scale_cov": True, "sparseGP": False, "model_groups": False},
    seed=SEED,
    outfile=str(output / "model.hdf5"),
)


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Loaded view='data' group='0' with N=8 samples and D=31 features...
Loaded view='data' group='1' with N=9 samples and D=31 features...
Loaded view='data' group='2' with N=8 samples and D=31 features...
Loaded view='data' group='3' with N=7 samples and D=31 features...
Loaded view='data' group='4' with N=22 samples and D=31 features...
Loaded view='data' group='5' with N=4 samples and D=

In [4]:
factors = np.asarray(adata.obsm["X_mofa"], float)
loadings = np.asarray(adata.varm["LFs"], float)
names = [f"factor_{i+1}" for i in range(factors.shape[1])]

sample_factors = metadata[["sample_id", "subject_id", "age", "country"]].copy()
sample_factors[names] = factors

subject_scores = (
    sample_factors.groupby("subject_id", as_index=False)
    .agg({**{name: "mean" for name in names}, "country": "first"})
)

feature_loadings = pd.DataFrame(loadings, columns=names)
feature_loadings.insert(0, "feature_id", counts.columns)

if len(names) != N_FACTORS:
    raise ValueError(f"Expected {N_FACTORS} factors; found {len(names)}.")

sample_factors.to_csv(output / "sample_factors.csv", index=False)
subject_scores.to_csv(output / "subject_scores.csv", index=False)
feature_loadings.to_csv(output / "feature_loadings.csv", index=False)
pd.DataFrame({
    "n_factors": [N_FACTORS],
    "n_iterations_requested": [N_ITERATIONS],
    "seed": [SEED],
}).to_csv(output / "settings.csv", index=False)

print("Saved:", output)
display(subject_scores.head())

Saved: ../data/mefisto/20260811_023339


,subject_id,factor_1,factor_2,factor_3,factor_4,factor_5,country
0,E002338,-0.273288,-0.308965,-0.235735,2.419470,1.196153,FIN
1,E002473,-0.247489,-0.820038,0.035321,2.421285,1.033282,FIN
2,E002681,0.270998,-0.420267,-0.122048,2.464544,1.695010,FIN
3,E003393,0.537726,-0.802617,-0.402778,2.465313,1.509227,FIN
4,E004071,-0.203352,-1.458269,0.857400,2.438942,1.258532,FIN
